
# 01 — Primary zero-shot benchmark and nomenclature sensitivity

> **Provenance note.** This is a cleaned **reference/reproducibility implementation reconstructed from the protocol documented in the paper**. It is not claimed to be the exact historical execution notebook. If the original executed notebook is available, prefer publishing that file (after removing credentials and machine-specific paths).

This notebook evaluates frozen **CLIP ViT-B/32**, **BioCLIP**, and **BioCLIP2** on the seven BFF-15/SylFishBD classes with the paper's four-template zero-shot ensemble. It also provides the BioCLIP2 Bengali-script and scientific-synonym controls.

Expected shared classes: Rui, Katla, Mrigal, Tilapia, Pabda, Ilish, Koi. No benchmark image is used for fitting or prompt optimization.


In [ ]:

from pathlib import Path
import json, math, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import open_clip
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


## Configuration
Set the two dataset roots. The discovery helper searches recursively and infers the fish class from path components.

In [ ]:

BFF_ROOT = Path('/path/to/BFF-15')
SYL_ROOT = Path('/path/to/SylFishBD')
OUT_DIR = Path('../results/generated_primary')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ['Rui','Katla','Mrigal','Tilapia','Pabda','Ilish','Koi']
EXPECTED_COUNTS = {
    'BFF-15': {'Rui':514,'Katla':427,'Mrigal':317,'Tilapia':383,'Pabda':348,'Ilish':233,'Koi':434},
    'SylFishBD': {'Rui':1670,'Katla':1133,'Mrigal':1293,'Tilapia':1326,'Pabda':862,'Ilish':789,'Koi':592},
}

NAMES = {
 'romanized': {'Rui':'Rui','Katla':'Katla','Mrigal':'Mrigal','Tilapia':'Tilapia','Pabda':'Pabda','Ilish':'Ilish','Koi':'Koi'},
 'english': {'Rui':'Rohu','Katla':'Catla','Mrigal':'Mrigal carp','Tilapia':'Nile tilapia','Pabda':'Pabda catfish','Ilish':'Hilsa','Koi':'Climbing perch'},
 'scientific': {'Rui':'Labeo rohita','Katla':'Catla catla','Mrigal':'Cirrhinus cirrhosus','Tilapia':'Oreochromis niloticus','Pabda':'Ompok pabda','Ilish':'Tenualosa ilisha','Koi':'Anabas testudineus'},
 'bengali': {'Rui':'রুই','Katla':'কাতলা','Mrigal':'মৃগেল','Tilapia':'তেলাপিয়া','Pabda':'পাবদা','Ilish':'ইলিশ','Koi':'কৈ'},
}
TEMPLATES = [
    'a photo of {name}, a fish species',
    'an image of {name}, a fish species',
    'a photograph of {name}, a fish species',
    'a specimen of {name}, a fish species',
]

IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}
ALIASES = {c.lower(): c for c in CLASSES}
ALIASES.update({'rohu':'Rui','catla':'Katla','mrigal carp':'Mrigal','nile tilapia':'Tilapia',
                'pabda catfish':'Pabda','hilsa':'Ilish','climbing perch':'Koi'})

def infer_class(path):
    parts = [p.lower().replace('_',' ').replace('-',' ').strip() for p in path.parts]
    for part in reversed(parts):
        if part in ALIASES: return ALIASES[part]
    for part in reversed(parts):
        for alias, cls in ALIASES.items():
            if alias in part: return cls
    return None

def discover(root, dataset):
    rows=[]
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            cls=infer_class(p.relative_to(root))
            if cls: rows.append((dataset, cls, str(p)))
    df=pd.DataFrame(rows, columns=['dataset','label','path'])
    print(dataset, len(df)); print(df.label.value_counts().reindex(CLASSES))
    return df

bff = discover(BFF_ROOT, 'BFF-15')
syl = discover(SYL_ROOT, 'SylFishBD')
manifest = pd.concat([bff,syl], ignore_index=True)
manifest.to_csv(OUT_DIR/'manifest.csv', index=False)


## Validate the exact paper subset

In [ ]:

for dataset, expected in EXPECTED_COUNTS.items():
    got = manifest[manifest.dataset.eq(dataset)].label.value_counts().to_dict()
    assert got == expected, f'{dataset}: expected {expected}, got {got}'
print('Exact paper counts validated:', len(manifest))


## Model loading and embedding helpers

In [ ]:

MODEL_SPECS = {
    'CLIP ViT-B/32': ('ViT-B-32', 'openai'),
    'BioCLIP': ('hf-hub:imageomics/bioclip', None),
    'BioCLIP2': ('hf-hub:imageomics/bioclip-2', None),
}

def load_model(display_name):
    model_name, pretrained = MODEL_SPECS[display_name]
    if model_name.startswith('hf-hub:'):
        model, preprocess = open_clip.create_model_from_pretrained(model_name, device=DEVICE)
    else:
        model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=DEVICE)
    tokenizer = open_clip.get_tokenizer(model_name)
    model.eval()
    return model, preprocess, tokenizer

@torch.inference_mode()
def encode_images(df, model, preprocess, batch_size=64):
    chunks=[]
    for start in tqdm(range(0,len(df),batch_size), desc='images'):
        batch=df.iloc[start:start+batch_size]
        ims=[preprocess(Image.open(p).convert('RGB')) for p in batch.path]
        x=torch.stack(ims).to(DEVICE)
        z=model.encode_image(x)
        z=z/z.norm(dim=-1,keepdim=True)
        chunks.append(z.cpu())
    return torch.cat(chunks).numpy()

@torch.inference_mode()
def class_prototypes(model, tokenizer, family_map, templates=TEMPLATES):
    protos=[]
    for cls in CLASSES:
        prompts=[t.format(name=family_map[cls]) for t in templates]
        tok=tokenizer(prompts).to(DEVICE)
        z=model.encode_text(tok)
        z=z/z.norm(dim=-1,keepdim=True)
        z=z.mean(0); z=z/z.norm()
        protos.append(z.cpu())
    return torch.stack(protos).numpy()

def predict(emb, protos):
    return np.asarray(CLASSES)[np.argmax(emb @ protos.T, axis=1)]

def metrics(y, pred):
    return {
      'accuracy_pct':100*accuracy_score(y,pred),
      'balanced_accuracy_pct':100*balanced_accuracy_score(y,pred),
      'macro_f1_pct':100*f1_score(y,pred,average='macro'),
    }


## Run the primary benchmark
Image embeddings are cached once per model; prompt-family comparisons modify only the text-side classifier.

In [ ]:

rows=[]
prediction_rows=[]
for model_name in MODEL_SPECS:
    print('\n###', model_name)
    model, preprocess, tokenizer = load_model(model_name)
    cache={}
    for dataset in ['BFF-15','SylFishBD']:
        d=manifest[manifest.dataset.eq(dataset)].reset_index(drop=True)
        cache[dataset]=encode_images(d, model, preprocess)
    families=['romanized','english','scientific'] + (['bengali'] if model_name=='BioCLIP2' else [])
    for dataset in ['BFF-15','SylFishBD']:
        d=manifest[manifest.dataset.eq(dataset)].reset_index(drop=True)
        emb=cache[dataset]
        for family in families:
            proto=class_prototypes(model, tokenizer, NAMES[family])
            pred=predict(emb, proto)
            m=metrics(d.label.to_numpy(), pred)
            rows.append({'model':model_name,'dataset':dataset,'prompt_family':family,**m})
            prediction_rows.extend({'model':model_name,'dataset':dataset,'prompt_family':family,
                                    'path':p,'label':y,'prediction':yp}
                                   for p,y,yp in zip(d.path,d.label,pred))
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

metrics_df=pd.DataFrame(rows)
preds_df=pd.DataFrame(prediction_rows)
metrics_df.to_csv(OUT_DIR/'primary_metrics.csv',index=False)
preds_df.to_csv(OUT_DIR/'primary_predictions.csv',index=False)
display(metrics_df)


## BioCLIP2 template and scientific-synonym sensitivity

In [ ]:

# Re-load BioCLIP2 and re-use this block if you want the template/synonym controls.
model, preprocess, tokenizer = load_model('BioCLIP2')

# Individual templates with the primary scientific names.
template_rows=[]
for dataset in ['BFF-15','SylFishBD']:
    d=manifest[manifest.dataset.eq(dataset)].reset_index(drop=True)
    emb=encode_images(d, model, preprocess)
    for template in TEMPLATES:
        proto=class_prototypes(model, tokenizer, NAMES['scientific'], templates=[template])
        pred=predict(emb, proto)
        template_rows.append({'dataset':dataset,'template':template,**metrics(d.label,pred)})

# One-name-at-a-time scientific variants.
variants = {
  'primary': {},
  'Katla=Gibelion catla': {'Katla':'Gibelion catla'},
  'Katla=Labeo catla': {'Katla':'Labeo catla'},
  'Mrigal=Cirrhinus mrigala': {'Mrigal':'Cirrhinus mrigala'},
}
syn_rows=[]
for dataset in ['BFF-15','SylFishBD']:
    d=manifest[manifest.dataset.eq(dataset)].reset_index(drop=True)
    emb=encode_images(d, model, preprocess)
    for variant, changes in variants.items():
        names=dict(NAMES['scientific']); names.update(changes)
        proto=class_prototypes(model, tokenizer, names)
        pred=predict(emb, proto)
        syn_rows.append({'dataset':dataset,'variant':variant,**metrics(d.label,pred)})

pd.DataFrame(template_rows).to_csv(OUT_DIR/'template_sensitivity.csv',index=False)
pd.DataFrame(syn_rows).to_csv(OUT_DIR/'scientific_synonym_sensitivity.csv',index=False)
display(pd.DataFrame(template_rows)); display(pd.DataFrame(syn_rows))



## Paper-value cross-check

After running, compare the generated aggregate results with `../results/primary_zero_shot_benchmark.csv`. Small deviations should be investigated rather than silently rounded away; preprocessing/checkpoint/library differences can affect frozen VLM scores.
